# Neural Collaborative Filtering (NCF)

## Objective

In this notebook, we build a **Neural Collaborative Filtering (NCF)** model using PyTorch.

Unlike traditional collaborative filtering methods that rely on similarity measures or matrix factorization, Neural Collaborative Filtering learns complex and non-linear relationships between users and movies through deep neural networks.

## Workflow

1. Load the processed MovieLens dataset.
2. Encode users and movies.
3. Create PyTorch datasets and dataloaders.
4. Build the Neural Collaborative Filtering model.
5. Train the model.
6. Evaluate the model.
7. Generate personalized movie recommendations.

## Why Neural Collaborative Filtering?

Traditional recommendation systems have limitations in capturing complex user preferences.

Neural Collaborative Filtering overcomes these limitations by learning dense user and movie embeddings jointly with multiple fully connected layers, enabling better recommendation quality.

This approach is widely used in modern recommendation systems such as Netflix, Amazon, YouTube, and Spotify.

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


## Import Required Libraries

The following libraries are used:

- **Pandas** for data manipulation.
- **NumPy** for numerical computation.
- **PyTorch** for building and training the neural recommendation model.
- **Train-Test Split** for evaluating the model.

In [2]:
# ==========================================
# Load Dataset
# ==========================================

ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=[
        "user_id",
        "movie_id",
        "rating",
        "timestamp"
    ]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    usecols=[0,1],
    names=[
        "movie_id",
        "movie_title"
    ]
)

movie_ratings = pd.merge(
    ratings,
    movies,
    on="movie_id"
)

print("Dataset Loaded Successfully")

movie_ratings.head()

Dataset Loaded Successfully


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


## Dataset Loading

The MovieLens 100K dataset contains:

- 943 Users
- 1,682 Movies
- 100,000 Ratings

Each record represents one interaction between a user and a movie.

In [3]:
# ==========================================
# Encode Users and Movies
# ==========================================

user_ids = movie_ratings["user_id"].unique()
movie_ids = movie_ratings["movie_id"].unique()

user_to_index = {
    user:i
    for i,user in enumerate(user_ids)
}

movie_to_index = {
    movie:i
    for i,movie in enumerate(movie_ids)
}

movie_ratings["user_encoded"] = movie_ratings["user_id"].map(user_to_index)

movie_ratings["movie_encoded"] = movie_ratings["movie_id"].map(movie_to_index)

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
0,196,242,3,881250949,Kolya (1996),0,0
1,186,302,3,891717742,L.A. Confidential (1997),1,1
2,22,377,1,878887116,Heavyweights (1994),2,2
3,244,51,2,880606923,Legends of the Fall (1994),3,3
4,166,346,1,886397596,Jackie Brown (1997),4,4


## Why Encoding?

Neural Networks cannot directly process categorical values like User IDs or Movie IDs.

Therefore, we convert each user and movie into a unique integer index.

These indices are later transformed into dense embedding vectors during training.

In [4]:
# ==========================================
# Train Test Split
# ==========================================

train_data, test_data = train_test_split(
    movie_ratings,
    test_size=0.2,
    random_state=42
)

print("Training Samples :", len(train_data))
print("Testing Samples :", len(test_data))

train_data.head()

Training Samples : 80000
Testing Samples : 20000


,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
75220,807,1411,1,893082619,Barbarella (1968),804,901
48955,474,659,5,887925187,Arsenic and Old Lace (1944),467,488
44966,463,268,4,877384940,Chasing Amy (1997),465,139
13568,139,286,4,879537844,"English Patient, The (1996)",321,289
92727,621,751,4,883799651,Tomorrow Never Dies (1997),618,261


## Train-Test Split

The dataset is divided into:

- 80% Training Data
- 20% Testing Data

The training data is used to learn user preferences, while the testing data is used to evaluate the model's prediction performance.

# Neural Collaborative Filtering (Deep Learning Recommendation System)

---

## Objective

In this notebook, we build a Deep Learning based Recommendation System using PyTorch.

Unlike traditional Collaborative Filtering, Neural Collaborative Filtering (NCF) learns complex non-linear relationships between users and movies.

The model learns dense embeddings for users and movies and predicts ratings using a neural network.

---

## Workflow

1. Load Processed Dataset
2. Encode Users & Movies
3. Train/Test Split
4. Create PyTorch Dataset
5. Build Neural Recommendation Model
6. Train the Model
7. Evaluate Performance
8. Save Trained Model

---

## Why Deep Learning?

Traditional methods:

- Popularity Based
- User-Based CF
- Item-Based CF
- Matrix Factorization

work well but have limitations.

Neural Networks can learn:

- Hidden user preferences
- Complex movie relationships
- Non-linear interactions
- Better latent representations

This approach is widely used in production recommendation systems.

In [5]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [6]:
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id","movie_id","rating","timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0,1],
    names=["movie_id","movie_title"]
)

movie_ratings = ratings.merge(
    movies,
    on="movie_id"
)

print("Dataset Loaded Successfully")

movie_ratings.head()

Dataset Loaded Successfully


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [7]:
user_to_index = {
    user: idx
    for idx, user in enumerate(movie_ratings["user_id"].unique())
}

movie_to_index = {
    movie: idx
    for idx, movie in enumerate(movie_ratings["movie_id"].unique())
}

movie_ratings["user_encoded"] = movie_ratings["user_id"].map(user_to_index)

movie_ratings["movie_encoded"] = movie_ratings["movie_id"].map(movie_to_index)

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
0,196,242,3,881250949,Kolya (1996),0,0
1,186,302,3,891717742,L.A. Confidential (1997),1,1
2,22,377,1,878887116,Heavyweights (1994),2,2
3,244,51,2,880606923,Legends of the Fall (1994),3,3
4,166,346,1,886397596,Jackie Brown (1997),4,4


In [8]:
train_data, test_data = train_test_split(
    movie_ratings,
    test_size=0.2,
    random_state=42
)

print("Training Samples :", len(train_data))
print("Testing Samples :", len(test_data))

train_data.head()

Training Samples : 80000
Testing Samples : 20000


,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
75220,807,1411,1,893082619,Barbarella (1968),804,901
48955,474,659,5,887925187,Arsenic and Old Lace (1944),467,488
44966,463,268,4,877384940,Chasing Amy (1997),465,139
13568,139,286,4,879537844,"English Patient, The (1996)",321,289
92727,621,751,4,883799651,Tomorrow Never Dies (1997),618,261


# Creating a Custom PyTorch Dataset

---

## Why do we need a Dataset class?

PyTorch models do not work directly with pandas DataFrames.

Instead, data must be converted into a format that PyTorch understands.

The `Dataset` class:

- Stores user IDs
- Stores movie IDs
- Stores ratings
- Returns one training example at a time

Each sample consists of:

- Encoded User ID
- Encoded Movie ID
- Rating

This makes it easy to load data efficiently during training using a DataLoader.

---

## Workflow

DataFrame

↓

Custom Dataset

↓

DataLoader

↓

Neural Network

In [9]:
# ==========================================
# Custom Dataset
# ==========================================

class MovieDataset(Dataset):

    def __init__(self, dataframe):

        self.users = dataframe["user_encoded"].values
        self.movies = dataframe["movie_encoded"].values
        self.ratings = dataframe["rating"].values.astype(np.float32)

    def __len__(self):

        return len(self.users)

    def __getitem__(self, idx):

        return (
            torch.tensor(self.users[idx], dtype=torch.long),
            torch.tensor(self.movies[idx], dtype=torch.long),
            torch.tensor(self.ratings[idx], dtype=torch.float32)
        )

## Why are we using Encoded IDs?

Neural Networks cannot process original user IDs or movie IDs directly.

For example:

Original User IDs

196
186
22
244

These values are random and have no meaningful order.

Instead, we convert them into sequential indices.

Example:

Original User ID → Encoded User ID

196 → 0

186 → 1

22 → 2

244 → 3

These encoded IDs are later used to look up trainable embedding vectors.

Embedding layers require indices ranging from **0 to N−1**, where **N** is the number of users or movies.

In [10]:
# ==========================================
# Create Dataset Objects
# ==========================================

train_dataset = MovieDataset(train_data)

test_dataset = MovieDataset(test_data)

print("Training Dataset Size :", len(train_dataset))
print("Testing Dataset Size  :", len(test_dataset))

Training Dataset Size : 80000
Testing Dataset Size  : 20000


## What is a Dataset Object?

A Dataset object is a wrapper around the original data.

Instead of storing an entire DataFrame inside the neural network, PyTorch accesses one sample at a time.

Each sample returned by the dataset contains:

- User ID
- Movie ID
- Rating

Example:

User → 25

Movie → 380

Rating → 4

This format is required for efficient mini-batch training.

In [11]:
# ==========================================
# Create DataLoaders
# ==========================================

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

print("Train Loader Batches :", len(train_loader))
print("Test Loader Batches  :", len(test_loader))

Train Loader Batches : 313
Test Loader Batches  : 79


# Why do we use a DataLoader?

Training a neural network on all 80,000 samples at once would require large memory and slow computation.

Instead, PyTorch divides the dataset into **mini-batches**.

Example:

Total Samples = 80,000

Batch Size = 256

Number of Batches ≈ 313

The model trains one batch at a time, making training:

- Faster
- Memory Efficient
- GPU Friendly
- Easier to Shuffle Data

The DataLoader automatically handles batching and shuffling during training.

In [12]:
# ==========================================
# Check One Batch
# ==========================================

users, movies, ratings = next(iter(train_loader))

print("Users Shape  :", users.shape)
print("Movies Shape :", movies.shape)
print("Ratings Shape:", ratings.shape)

print("\nFirst 10 Users:")
print(users[:10])

print("\nFirst 10 Movies:")
print(movies[:10])

print("\nFirst 10 Ratings:")
print(ratings[:10])

Users Shape  : torch.Size([256])
Movies Shape : torch.Size([256])
Ratings Shape: torch.Size([256])

First 10 Users:
tensor([ 48, 613, 770, 542, 531, 254,  10, 376, 781, 802])

First 10 Movies:
tensor([  68,  528,  911, 1084,  137,  334,  161,  950,  153,  433])

First 10 Ratings:
tensor([4., 5., 3., 4., 3., 3., 5., 3., 4., 2.])


# Building the Neural Collaborative Filtering (NCF) Model

---

## Objective

Now that the data has been prepared, we will build a Deep Learning recommendation model.

The model learns two types of embeddings:

- User Embeddings
- Movie Embeddings

These embeddings are dense vector representations that capture hidden patterns and preferences.

Instead of manually calculating similarities (as in User-Based or Item-Based Collaborative Filtering), the neural network learns these relationships automatically during training.

---

## Model Architecture

User ID
      │
      ▼
User Embedding
      │

Movie ID
      │
      ▼
Movie Embedding
      │

Concatenate
      │
      ▼
Fully Connected Layer
      │
      ▼
ReLU Activation
      │
      ▼
Dropout
      │
      ▼
Fully Connected Layer
      │
      ▼
Predicted Rating

---

## Why Embeddings?

Instead of representing each user using thousands of movie ratings, we compress each user into a small dense vector.

For example,

User 196

↓

[0.84, -1.21, 0.52, ..., 0.17]

Similarly, every movie also gets its own embedding vector.

These vectors are learned automatically during training and capture latent user preferences and movie characteristics.

In [13]:
# ==========================================
# Neural Collaborative Filtering Model
# ==========================================

class NCF(nn.Module):

    def __init__(self, num_users, num_movies, embedding_dim=50):

        super().__init__()

        # User Embedding Layer
        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        # Movie Embedding Layer
        self.movie_embedding = nn.Embedding(
            num_movies,
            embedding_dim
        )

        # Fully Connected Layers
        self.fc1 = nn.Linear(
            embedding_dim * 2,
            128
        )

        self.fc2 = nn.Linear(
            128,
            64
        )

        self.output = nn.Linear(
            64,
            1
        )

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.2)

    def forward(self, user, movie):

        user_vec = self.user_embedding(user)

        movie_vec = self.movie_embedding(movie)

        x = torch.cat(
            [user_vec, movie_vec],
            dim=1
        )

        x = self.relu(self.fc1(x))

        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        prediction = self.output(x)

        return prediction.squeeze()

## Understanding the Model Architecture

The model consists of the following components:

### 1. User Embedding Layer

Converts each encoded user ID into a dense vector.

Example:

User ID → Embedding

15 → [0.32, -0.74, 0.51, ...]

---

### 2. Movie Embedding Layer

Converts each encoded movie ID into another dense vector.

Movie ID → Embedding

235 → [1.42, -0.29, 0.18, ...]

---

### 3. Concatenation

The user embedding and movie embedding are joined together into one feature vector.

Example:

User Embedding (50)

+

Movie Embedding (50)

↓

Combined Vector (100)

---

### 4. Fully Connected Layers

These layers learn complex interactions between users and movies.

Unlike matrix factorization, neural networks can model highly non-linear relationships.

---

### 5. Output Layer

Produces a single predicted rating for the given user-movie pair.

Example:

Predicted Rating

4.37

In [14]:
# ==========================================
# Create Neural Network
# ==========================================

num_users = movie_ratings["user_encoded"].nunique()

num_movies = movie_ratings["movie_encoded"].nunique()

model = NCF(
    num_users,
    num_movies,
    embedding_dim=50
)

print(model)

NCF(
  (user_embedding): Embedding(943, 50)
  (movie_embedding): Embedding(1682, 50)
  (fc1): Linear(in_features=100, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)


## Why are there 943 Users and 1682 Movies?

From our dataset:

- Unique Users = 943
- Unique Movies = 1682

Each user receives a trainable embedding vector of length 50.

Therefore,

User Embedding Matrix

943 × 50

Similarly,

Movie Embedding Matrix

1682 × 50

These embeddings are initialized randomly and are optimized during training using backpropagation.

By the end of training, users with similar preferences and movies with similar characteristics will have similar embedding vectors.

# Training the Neural Collaborative Filtering Model

---

## Objective

The Neural Collaborative Filtering (NCF) model has been created successfully.

The next step is to train the model so that it can learn user preferences from historical ratings.

Training consists of three main components:

1. Loss Function
2. Optimizer
3. Training Loop

During training:

- A batch of user IDs and movie IDs is passed to the model.
- The model predicts ratings.
- The predicted ratings are compared with the true ratings.
- The prediction error (loss) is calculated.
- The optimizer updates the model parameters using backpropagation.

This process is repeated for multiple epochs until the model learns meaningful user and movie embeddings.

---

## Training Pipeline

Training Data

↓

Neural Network

↓

Predicted Ratings

↓

Loss Function

↓

Backpropagation

↓

Update Weights

↓

Repeat for Multiple Epochs

In [15]:
# ==========================================
# Loss Function and Optimizer
# ==========================================

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss Function :", criterion)
print("Optimizer :", optimizer)

Loss Function : MSELoss()
Optimizer : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Why do we use Mean Squared Error (MSE)?

Our task is a regression problem because we predict movie ratings.

Example:

Actual Rating = 5

Predicted Rating = 4.2

Error = 0.8

MSE squares this error and averages it over all samples.

The objective of training is to minimize this error.

---

## Why Adam Optimizer?

Adam is one of the most widely used optimization algorithms in Deep Learning.

Advantages:

- Fast convergence
- Adaptive learning rate
- Stable training
- Works well for recommendation systems

For this project we use:

Learning Rate = 0.001

In [16]:
# ==========================================
# Train Neural Network
# ==========================================

num_epochs = 10

train_losses = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    for users, movies, ratings in train_loader:

        optimizer.zero_grad()

        predictions = model(users, movies)

        loss = criterion(predictions, ratings)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    train_losses.append(epoch_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f}"
    )

Epoch [1/10] Loss: 1.8825
Epoch [2/10] Loss: 1.0956
Epoch [3/10] Loss: 1.0186
Epoch [4/10] Loss: 0.9742
Epoch [5/10] Loss: 0.9465
Epoch [6/10] Loss: 0.9255
Epoch [7/10] Loss: 0.9085
Epoch [8/10] Loss: 0.8941
Epoch [9/10] Loss: 0.8810
Epoch [10/10] Loss: 0.8709


## What happens during each Epoch?

An epoch means the model has seen the **entire training dataset once**.

For each epoch:

1. The training data is divided into mini-batches.
2. The model predicts ratings for each batch.
3. The prediction error (loss) is calculated.
4. Gradients are computed using backpropagation.
5. The optimizer updates the model parameters.
6. The process repeats until all batches are processed.

As training progresses, the loss should gradually decrease, indicating that the model is learning meaningful patterns from the data.

# Model Training Completed

The Neural Collaborative Filtering (NCF) model has been trained for 10 epochs using the Adam optimizer and Mean Squared Error (MSE) loss.

## Training Summary

- Optimizer: Adam
- Learning Rate: 0.001
- Loss Function: Mean Squared Error (MSE)
- Epochs: 10
- Batch Size: 256

## Training Loss

The training loss consistently decreased during training.

| Epoch | Loss |
|-------:|------:|
|1|1.8825|
|2|1.0956|
|3|1.0186|
|4|0.9742|
|5|0.9465|
|6|0.9255|
|7|0.9085|
|8|0.8941|
|9|0.8810|
|10|0.8709|

## Observation

The decreasing loss indicates that the model is successfully learning user-item interaction patterns.

The learned embeddings now capture latent preferences of users and movies, enabling personalized recommendation generation.

The next step is to evaluate the trained model on unseen test data.

In [17]:
# ==========================================
# Evaluate Model on Test Dataset
# ==========================================

model.eval()

test_loss = 0

with torch.no_grad():

    for users, movies, ratings in test_loader:

        predictions = model(users, movies)

        loss = criterion(predictions, ratings)

        test_loss += loss.item()

average_test_loss = test_loss / len(test_loader)

print("Average Test Loss :", round(average_test_loss, 4))

Average Test Loss : 0.9163


# Model Evaluation

After training, the Neural Collaborative Filtering (NCF) model was evaluated on the unseen test dataset.

## Test Performance

**Average Test Loss:** `0.9163`

The test loss measures how accurately the model predicts movie ratings for user-movie interactions that were not seen during training.

A lower test loss indicates better generalization and stronger recommendation capability.

## Interpretation

- The training loss continuously decreased during training.
- The test loss is close to the final training loss.
- This suggests that the model has learned meaningful user-item interaction patterns without severe overfitting.

Overall, the model demonstrates good predictive performance on unseen data and is ready to generate personalized movie recommendations.

In [18]:
# ==========================================
# Save Trained NCF Model
# ==========================================

import os
import torch

os.makedirs("models", exist_ok=True)

torch.save(
    model.state_dict(),
    "models/ncf_model.pth"
)

print("NCF Model Saved Successfully!")

NCF Model Saved Successfully!


In [19]:
# ==========================================
# Save Learned Embeddings
# ==========================================

import pandas as pd

# User Embeddings
user_embeddings = model.user_embedding.weight.detach().cpu().numpy()

user_embeddings_df = pd.DataFrame(user_embeddings)

user_embeddings_df.to_csv(
    "models/ncf_user_embeddings.csv",
    index=False
)

# Movie Embeddings
movie_embeddings = model.movie_embedding.weight.detach().cpu().numpy()

movie_embeddings_df = pd.DataFrame(movie_embeddings)

movie_embeddings_df.to_csv(
    "models/ncf_movie_embeddings.csv",
    index=False
)

print("Embeddings Saved Successfully!")

Embeddings Saved Successfully!


In [20]:
import os

print(os.listdir("models"))

['item_mapping.csv', 'ncf_movie_embeddings.csv', 'item_embeddings.csv', 'user_faiss.index', 'ncf_user_embeddings.csv', 'ncf_model.pth', 'user_mapping.csv', 'user_embeddings.csv']
